# Submission Generation

## Objective

Generate the Kaggle submission file using the pre-built RAG pipeline.

This notebook performs the following:

1. Bootstrap the project
2. Load configuration
3. Load datasets
4. Build the Knowledge Base
5. Load precomputed embeddings
6. Load the FAISS index
7. Create the Retrieval Pipeline
8. Generate Top-3 predictions
9. Create submission.csv

No training is performed in this notebook.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from tqdm.auto import tqdm

from src.config import (
    load_config,
    get_embedding_model,
    get_top_k,
    get_reranker_model,
)

from src.knowledge_base import KnowledgeBaseService
from src.embeddings import EmbeddingService
from src.faiss_index import FaissIndexService
from src.retriever import RetrieverService
from src.reranker import RerankerService
from src.rag_pipeline import RAGPipeline


In [2]:
config = load_config()

embedding_model_name = get_embedding_model(config)
reranker_model_name = get_reranker_model(config)
top_k = get_top_k(config)

print(f"Embedding Model : {embedding_model_name}")
print(f"Reranker Model  : {reranker_model_name}")
print(f"Top K           : {top_k}")

Embedding Model : all-MiniLM-L6-v2
Reranker Model  : cross-encoder/ms-marco-MiniLM-L-6-v2
Top K           : 3


In [3]:
train = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "train.csv"
)

test = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "test.csv"
)

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [4]:
knowledge_base_service = KnowledgeBaseService()

knowledge_base_service.build(train)

print(
    f"Knowledge Base Size : "
    f"{len(knowledge_base_service.get_documents())}"
)

2026-07-17 15:28:31,792 | INFO | src.knowledge_base | Building Knowledge Base...
2026-07-17 15:28:31,807 | INFO | src.knowledge_base | Knowledge Base built successfully with 2000 documents.


Knowledge Base Size : 2000


In [5]:
embedding_service = EmbeddingService(
    embedding_model_name
)

embedding_service.load(
    PROJECT_ROOT
    / "outputs"
    / "embeddings"
    / "knowledge_base_embeddings.npy"
)

2026-07-17 15:28:31,810 | INFO | src.embeddings | Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-17 15:28:37,796 | INFO | src.embeddings | Embedding model loaded.
2026-07-17 15:28:37,797 | INFO | src.embeddings | Loading embeddings from /Users/nishantskuttan/code/dlgenaiproject_may_2026/outputs/embeddings/knowledge_base_embeddings.npy


array([[-0.09946356,  0.00865054, -0.04990312, ..., -0.02489339,
         0.021535  ,  0.02436984],
       [-0.07254623,  0.03709433, -0.02181764, ..., -0.03332438,
         0.06457467,  0.02372232],
       [-0.00166186, -0.00811861, -0.00320865, ..., -0.02536879,
        -0.06575194,  0.02632993],
       ...,
       [-0.01881237,  0.03754336, -0.07955373, ..., -0.00745284,
        -0.03025148,  0.00373754],
       [-0.02653124, -0.0369733 ,  0.12620695, ...,  0.01014461,
        -0.03782934, -0.02695566],
       [ 0.03329898, -0.02667525, -0.02392747, ...,  0.05930161,
         0.04234194, -0.01237911]], shape=(2000, 384), dtype=float32)

In [6]:
faiss_service = FaissIndexService()

faiss_service.load(
    PROJECT_ROOT
    / "outputs"
    / "faiss"
    / "knowledge_base.faiss"
)

2026-07-17 15:28:37,810 | INFO | src.faiss_index | FAISS index loaded successfully.


In [7]:
retriever_service = RetrieverService(
    embedding_service,
    faiss_service,
)

In [8]:
reranker_service = RerankerService(
    reranker_model_name
)

2026-07-17 15:28:37,839 | INFO | src.reranker | Loading Cross Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-07-17 15:28:46,718 | INFO | src.reranker | Cross Encoder loaded successfully.


In [9]:
rag_pipeline = RAGPipeline(
    knowledge_base_service,
    retriever_service,
    reranker_service,
)

In [10]:
sample_prediction = rag_pipeline.predict(
    test.iloc[0],
    top_k=top_k,
)

print(sample_prediction)

2026-07-17 15:28:46,742 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 15:28:46,743 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 15:28:46,743 | INFO | src.retriever | Embedding query.
2026-07-17 15:28:46,744 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 15:28:47,128 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 15:28:47,128 | INFO | src.retriever | Searching FAISS index.
2026-07-17 15:28:47,131 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 15:28:47,131 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 15:28:47,446 | INFO | src.reranker | Reranking completed.
2026-07-17 15:28:47,447 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 15:28:47,447 | INFO | src.rag_pipeline | Prediction completed.


['A', 'B', 'C']


In [11]:
predictions = []

for row in tqdm(
    test.itertuples(index=False),
    total=len(test),
    desc="Generating predictions",
):

    prediction = rag_pipeline.predict(
        row._asdict(),
        top_k=top_k,
    )

    predictions.append(
        " ".join(prediction)
    )

Generating predictions:   0%|          | 0/500 [00:00<?, ?it/s]

2026-07-17 15:28:47,454 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 15:28:47,454 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 15:28:47,454 | INFO | src.retriever | Embedding query.
2026-07-17 15:28:47,455 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 15:28:47,464 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 15:28:47,465 | INFO | src.retriever | Searching FAISS index.
2026-07-17 15:28:47,465 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 15:28:47,465 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 15:28:47,478 | INFO | src.reranker | Reranking completed.
2026-07-17 15:28:47,478 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 15:28:47,479 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 15:28:47,479 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 15:28:47,479 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 15:28:47,480 | INFO | src.retriever | Embedding query.
2026-07-17 15:28:47,480 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

2026-07-17 15:29:03,828 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 15:29:03,828 | INFO | src.retriever | Searching FAISS index.
2026-07-17 15:29:03,828 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 15:29:03,828 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 15:29:03,838 | INFO | src.reranker | Reranking completed.
2026-07-17 15:29:03,838 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 15:29:03,839 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 15:29:03,839 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 15:29:03,839 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 15:29:03,839 | INFO | src.retriever | Embedding query.
2026-07-17 15:29:03,839 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 15:29:03,851 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 15:29:03,851 | INFO | src.retriever | Searching FAISS index.
2026-07-17 15:29:03,851 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 15:29:03,852 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 15:29:03,862 | INFO | src.reranker | Reranking completed.
2026-07-17 15:29:03,862 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 15:29:03,862 | INFO | src.rag_pipeline | Prediction completed.
2026-07-17 15:29:03,862 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-17 15:29:03,862 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-17 15:29:03,863 | INFO | src.retriever | Embedding query.
2026-07-17 15:29:03,863 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-17 15:29:03,870 | INFO | src.embeddings | Embedding generation completed.
2026-07-17 15:29:03,871 | INFO | src.retriever | Searching FAISS index.
2026-07-17 15:29:03,871 | INFO | src.retriever | Retrieved 3 documents.
2026-07-17 15:29:03,871 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-17 15:29:03,882 | INFO | src.reranker | Reranking completed.
2026-07-17 15:29:03,882 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-17 15:29:03,882 | INFO | src.rag_pipeline | Prediction completed.


In [12]:
submission = pd.DataFrame(
    {
        "id": test["id"],
        "prediction": predictions,
    }
)

submission.head()

,id,prediction
0,1,A B C
1,2,A B C
2,3,B A C
3,4,E A B
4,5,C A B


In [14]:
assert len(submission) == len(test)

assert submission["prediction"].notna().all()

assert submission["prediction"].str.count(" ").eq(2).all()

print("Submission validated successfully.")

Submission validated successfully.
